In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from Session9_code.utils import db_helpers as utils

In [ ]:
_ = load_dotenv()

In [ ]:
llm = init_chat_model(model='gpt-4o-mini',temperature=0)

In [ ]:
# Generate Database
# utils.create_transactions_db()

In [ ]:
# Show DB Schema
utils.print_html(utils.get_schema('products.db'))

In [ ]:
# Question to SQL Query Method

from langchain_core.messages import HumanMessage


def generate_sql(question: str, schema: str) -> str:
    """سؤال کاربر را دریافت کرده و یک کوئری SQL برای SQLite تولید می‌کند."""
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [ ]:
schema = utils.get_schema('products.db')

question = 'کدام رنگ بیشترین مجموع فروش رو داشته؟'

sql_v1 = generate_sql(question=question,schema=schema)

utils.print_html(content=question,title="User Question")
utils.print_html(content=sql_v1, title="Generated SQL Query (V1)")

# Reflection

In [ ]:
import json


def refine_sql(
    question: str,
    sql_query: str,
    schema: str,
 ) -> tuple[str, str]:
    """
    بررسی می‌کند که آیا کوئری به سؤال پاسخ می‌دهد و در صورت نیاز آن را اصلاح می‌کند.
    خروجی: (feedback, refined_sql)
    """
    prompt = f"""
You are a SQL reviewer and refiner.

User asked:
{question}

Original SQL:
{sql_query}

Table Schema:
{schema}

Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
Step 2: If improvement is needed, provide a refined SQL query for SQLite.
If the original SQL is already correct, return it unchanged.

Return STRICT JSON with two fields:
{{
  "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
  "refined_sql": "<final SQL to run>"
}}
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()

    # حذف backtick های احتمالی markdown
    content = content.replace("```json", "").replace("```", "").strip()

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception  :  # noqa: BLE001
        feedback = content
        refined_sql = sql_query

    return feedback, refined_sql

In [ ]:
feedback, sql_v2_no_ext = refine_sql(
    question=question,
    sql_query=sql_v1,
    schema=schema)

# نمایش V1
utils.print_html(question, title="User Question")
utils.print_html(sql_v1, title="Generated SQL Query (V1)")

# اجرای V1 برای نشان دادن مشکل
df_sql_v1 = utils.execute_sql(sql_v1, db_path='products.db')
utils.print_html(df_sql_v1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# نمایش feedback و V2
utils.print_html(feedback, title="Feedback on V1 (without external feedback)")
utils.print_html(sql_v2_no_ext, title="Refined SQL Query (V2 - no external feedback)")

In [ ]:
question = 'ارزان ترین محصول ها کدوم ها هستند؟'
feedback, sql_v3_no_ext = refine_sql(
    question=question,
    sql_query=sql_v1,
    schema=schema)

# نمایش V1
utils.print_html(question, title="User Question")
utils.print_html(sql_v1, title="Generated SQL Query (V1)")

# اجرای V1 برای نشان دادن مشکل
df_sql_v1 = utils.execute_sql(sql_v1, db_path='products.db')
utils.print_html(df_sql_v1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# نمایش feedback و V2
utils.print_html(feedback, title="Feedback on V1 (without external feedback)")
utils.print_html(sql_v3_no_ext, title="Refined SQL Query (V2 - no external feedback)")

<div style="border:1px solid #fecaca; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ **نکته مهم:**  
حتی اگر کوئری از نظر متنی درست به نظر برسد و feedback هم آن را تأیید کند، **نتیجه اجرای واقعی** ممکن است مشکل پنهانی داشته باشد.  

در اینجا `qty_delta` برای فروش **منفی** است، پس `qty_delta * unit_price` یک عدد منفی می‌شود — مشکلی که با خواندن متن کوئری قابل تشخیص نیست.  

👉 به همین دلیل، agent باید **خروجی اجرا** را هم بررسی کند.

</div>

#### 3.2.2. Final Approach: Refine an SQL Query with External Feedback

برخلاف مرحله قبل که مدل فقط **متن کوئری** را بررسی می‌کرد، در اینجا **نتیجه واقعی اجرا** را به‌عنوان بازخورد خارجی ارائه می‌دهیم.  
این بازخورد به مدل اجازه می‌دهد تا مشکلات واقعی (مانند اعداد منفی، فیلترهای ناقص یا خطاهای گروه‌بندی) را شناسایی کند.

In [ ]:
import pandas as pd


def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
) -> tuple[str, str]:
    """
    با استفاده از نتیجه واقعی اجرای SQL (بازخورد خارجی)، کوئری را ارزیابی و اصلاح می‌کند.
    خروجی: (feedback, refined_sql)
    """
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_query}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Table Schema:
    {schema}

    Step 1: Briefly evaluate if the SQL output answers the user's question.
    Step 2: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()

    # حذف backtick های احتمالی markdown
    content = content.replace("```json", "").replace("```", "").strip()

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        feedback = content
        refined_sql = sql_query

    return feedback, refined_sql

In [ ]:
question = 'کدام رنگ بیشترین مجموع فروش رو داشته؟'


# اجرای کوئری اولیه (V1)
df_sql_v1 = utils.execute_sql(sql_v1, db_path='products.db')

# اصلاح با بازخورد خارجی (نتیجه واقعی اجرا)
feedback, sql_v2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_v1,
    df_feedback=df_sql_v1,
    schema=schema,
)

# نمایش V1
utils.print_html(question, title="User Question")
utils.print_html(sql_v1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_v1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# نمایش feedback و V2
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_v2, title="Refined SQL Query (V2)")

# اجرا و نمایش نتیجه V2
df_sql_v2 = utils.execute_sql(sql_v2, db_path='products.db')
utils.print_html(df_sql_v2, title="SQL Output of V2 (with External Feedback) - ✅ Fully answers the question")

# Full Workflow Steps for SQL Reflection

In [ ]:
def run_sql_workflow(
    db_path: str,
    question: str,
):
    """
    Pipeline کامل برای تولید، اجرا، ارزیابی و اصلاح کوئری SQL.

    مراحل:
      1) استخراج schema
      2) تولید SQL (V1)
      3) اجرای V1 → نمایش خروجی
      4) Reflection با بازخورد خارجی → SQL اصلاح‌شده (V2)
      5) اجرای V2 → پاسخ نهایی
    """

    # گام 1: استخراج schema
    schema = utils.get_schema(db_path)
    utils.print_html(schema, title="📘 Step 1 — Extract Database Schema")

    # گام 2: تولید SQL (V1)
    sql_v1 = generate_sql(question, schema)
    utils.print_html(sql_v1, title="🧠 Step 2 — Generate SQL (V1)")

    # گام 3: اجرای V1
    df_v1 = utils.execute_sql(sql_v1, db_path)
    utils.print_html(df_v1, title="🧪 Step 3 — Execute V1 (SQL Output)")

    # گام 4: Reflection با بازخورد خارجی → V2
    feedback, sql_v2 = refine_sql_external_feedback(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,
        schema=schema,
    )
    utils.print_html(feedback, title="🧭 Step 4 — Reflect on V1 (Feedback)")
    utils.print_html(sql_v2, title="🔁 Step 4 — Refined SQL (V2)")

    # گام 5: اجرای V2
    df_v2 = utils.execute_sql(sql_v2, db_path)
    utils.print_html(df_v2, title="✅ Step 5 — Execute V2 (Final Answer)")

In [ ]:
run_sql_workflow(
    "products.db",
    "Which color of product has the highest total sales?",
)